# From Look-Alikes to Action: A Case File for Maria
**INFO 5810 – Data Analysis and Knowledge Discovery**

**What you will be able to do after this notebook**
- Turn free-text notes into numbers and **retrieve similar claims** by hand (the "R" in RAG).
- **Cluster** claims into groups with a hand-built k-means, and read what the groups say and do not say.
- Build a **knowledge graph** of claims, shops, areas and adjusters, and ask it questions in NetworkX and in Neo4j (Cypher).
- Turn what you find into **hypotheses**, and **test them causally** with the graph, adjustment and do().
- Answer a question about **one person** with a counterfactual and recourse.
- Treat Maria as a **customer**, not only a claim: keep a customer record and an interaction log (the heart of **CRM**), measure what a false alarm costs in renewals, and test a better next step with a pilot.
- Put it all together in a **case file** that says what each piece of evidence can and cannot support.

**The thread through the whole notebook.** Everything we build sits on the ladder. Similarity, clusters and graph structure are **rung 1**: they find what goes with what. Testing a hypothesis about an action is **rung 2**. Maria's own plan is **rung 3**. The skill is knowing which rung each finding lives on, and not writing a rung 1 finding up as a cause.

**A second thread: Maria is a customer.** Every claim belongs to a person who may or may not renew next year. Customer relationship management (CRM) keeps one record of each customer and every contact with them, and uses it to decide what to do next. We add that layer to the same case, so the analysis ends in something an insurer can act on and measure.

## Part 0: Where we have been (quick recap)

Almost everything today reuses an earlier idea. Here is the short version.

| Idea we already know | One-line reminder | Where it comes back today |
|---|---|---|
| **The ladder** | Seeing needs data. Doing needs data plus a graph. Imagining one case needs data, a graph and equations. | Organizes the whole case file |
| **A model that flags claims** | A logistic regression turns four inputs into a fraud score, and flags at 0.5 | Maria is flagged again, and we ask what to do about it |
| **False positive** | A legitimate claim the model flags | Maria is one |
| **TF-IDF and cosine similarity** | Count words, give rare words more weight, compare notes by angle | Retrieving claims that read like Maria's |
| **Graph databases** | Nodes, edges, and queries that follow them (Cypher) | The knowledge graph of claims, shops and areas |
| **Graph RAG** | Retrieve facts first, then ground the answer in exactly those facts | The case file is built from retrieved facts |
| **Fork, confounder, backdoor path** | A common cause makes two things move together; adjust for it | Why a shop can look guilty without being guilty |
| **do() and adjustment** | Force a variable, or hold the confounder fixed | Testing whether a shop really raises fraud |
| **Structural causal model** | Equations plus each case's own noise | Maria's replay |
| **Counterfactual and recourse** | What would it take for *this* person, using only actions she can take | Maria's plan |

**What CRM means here.** *Customer relationship management* is the practice of keeping one joined-up record of each customer and every contact with them, then using it to decide what to do next. Each piece of CRM has a counterpart in this notebook:

| CRM idea | What it is in this notebook |
|---|---|
| **Customer record** | Tenure, annual premium and renewal outcome for each claimant |
| **Interaction log** | Every call, document request, review and settlement, stored as a time-stamped record on the claim |
| **Segments** | The claim groups from clustering, read as groups of customers |
| **Customers like this one** | Retrieval over the adjuster notes |
| **Relationship map** | The knowledge graph that links customers, claims, shops and adjusters |
| **Customer 360** | One query that returns everything we know about one customer |
| **Next best action** | The recommended step for Maria (the case file) |
| **Closed loop** | Log the action, test it with a pilot, learn from the result |

**The ladder, one more time**

| Rung | Kind of question | What you need |
|---|---|---|
| 1. **Seeing** | What goes with what? | Data |
| 2. **Doing** | What happens to a group if we *make* something happen? | Data and a graph |
| 3. **Imagining** | What would have happened to *this* case if something had been different? | Data, a graph and equations |

### Setup
The first cell checks that the Neo4j driver is installed (run `uv sync` once in the project folder if it is not). The next one reads your connection settings from a `.env` file (the same three names as the earlier graph demo: `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`) or from your environment. **If no Neo4j password is found, every Neo4j cell is skipped and the notebook still runs end to end with NetworkX.**

In [ ]:
try:
    from neo4j import GraphDatabase
except ImportError as e:
    raise ImportError("The neo4j driver is missing. Run `uv sync` in the project folder, then select the .venv kernel.") from e

In [ ]:
import os, re, json, warnings, logging
from pathlib import Path
warnings.filterwarnings("ignore")
logging.disable(logging.CRITICAL)

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass                      # no python-dotenv: set NEO4J_PASSWORD in your environment instead

NEO4J_URI      = os.environ.get("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER     = os.environ.get("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD")
GROQ_API_KEY   = os.environ.get("GROQ_API_KEY")          # optional, used only in the last part

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
pd.set_option("display.max_colwidth", 120)

NEO4J_OK, driver = False, None
if NEO4J_PASSWORD:
    try:
        driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
        driver.verify_connectivity()
        NEO4J_OK = True
        print(f"Connected to Neo4j at {NEO4J_URI}")
    except Exception as e:
        print("Could not connect to Neo4j:", type(e).__name__, "-", str(e)[:140])
else:
    print("No NEO4J_PASSWORD found in .env or the environment.")
if not NEO4J_OK:
    print("Neo4j cells will be skipped. Everything else runs with NetworkX.")

def run_cypher(query, **params):
    with driver.session() as session:
        return [record.data() for record in session.run(query, **params)]

def show_cypher(query, **params):
    print(query.strip())
    if not NEO4J_OK:
        print("\n(Neo4j not connected: skipped. The NetworkX answer above asks the same question.)")
        return None
    return pd.DataFrame(run_cypher(query, **params))

## Part 1: Maria's false alarm, and what the insurer does next

### The story so far
The insurer's model flagged **Maria's** claim for an extra fraud review. A full review later confirmed that her claim was **legitimate**. The flag was a false alarm, a **false positive**.

Maria was told two things: *why* she was flagged, and *what would have helped her*. Now the insurer's analysts ask bigger questions about the same case:

1. **Who else looks like Maria?** Did claims that read like hers turn out to be fraud?
2. **What network is her claim part of?** Which shop, which area, which adjuster?
3. **Is any of that a cause?** If the network looks risky, should the insurer act on the network?
4. **What does Maria herself need?**
5. **What did the flag cost the insurer as a business?** Maria had been a customer for six years. Did she stay?

We will build a **case file** that answers all four, one rung at a time.

| Part of the case file | Tool | Rung | Question |
|---|---|---|---|
| Similar past claims | Similarity over adjuster notes (retrieval) | 1 | What did claims like hers turn out to be? |
| Groups of claims | Clustering | 1 | Which group does she fall into? |
| Her network | Knowledge graph (NetworkX and Neo4j) | 1 | Who and what is she connected to? |
| A hypothesis check | Graph, adjustment, do() | 2 | Is a pattern in the network a cause? |
| Her own plan | Counterfactual and recourse | 3 | What would it take for *her*? |
| Customers and retention | Customer record, interaction log, adjustment | 1 and 2 | Does a false alarm cost us customers? |
| Her renewal | Counterfactual | 3 | Would she have stayed if she had not been flagged? |
| Closing the loop | Randomized pilot | 2 | Does the next best action work? |

### The model, rebuilt
This is the same model as before: a logistic regression trained on 40,000 past claims, with Maria's file unchanged. The next two cells rebuild it. You have seen this machine, so we do not open it again.

In [ ]:
FRAUD_P = {(1, 1): 0.90, (1, 0): 0.55, (0, 1): 0.50, (0, 0): 0.03}   # P(FraudRisk = High | prior, timing)

def eq_receipts(fraud, docs, u):  return 4.0 + 3.0 * fraud - 0.5 * docs + u
def eq_witness(fraud, follow, u): return 1.0 + 3.0 * fraud - 1.5 * follow + u

def draw_noise(n, rng):
    return dict(prior=rng.uniform(size=n), timing=rng.uniform(size=n), fraud=rng.uniform(size=n),
                docs=rng.normal(0, 1.5, n), follow=rng.uniform(size=n),
                receipts=rng.normal(0, 1.2, n), witness=rng.normal(0, 1.5, n))

def world(U, do=None, docs_effect=-0.5):
    do = do or {}
    d = {}
    d["PriorClaims"]      = do.get("PriorClaims", (U["prior"] < 0.15).astype(int))
    d["SuspiciousTiming"] = do.get("SuspiciousTiming", (U["timing"] < 0.10).astype(int))
    p = np.vectorize(lambda a, b: FRAUD_P[(int(a), int(b))])(d["PriorClaims"], d["SuspiciousTiming"])
    d["FraudRisk"]        = do.get("FraudRisk", (U["fraud"] < p).astype(int))
    d["Docs"]             = do.get("Docs", 4 + 3.0 * d["PriorClaims"] + U["docs"])
    d["FollowUp"]         = do.get("FollowUp", (U["follow"] < 0.10).astype(int))
    d["Receipts"]         = do.get("Receipts", 4.0 + 3.0 * d["FraudRisk"] + docs_effect * d["Docs"] + U["receipts"])
    d["Witness"]          = do.get("Witness", eq_witness(d["FraudRisk"], d["FollowUp"], U["witness"]))
    return pd.DataFrame(d)

history = world(draw_noise(40000, np.random.default_rng(13)))
features = ["PriorClaims", "SuspiciousTiming", "Receipts", "Witness"]
model = LogisticRegression(max_iter=2000).fit(history[features], history["FraudRisk"])

In [ ]:
maria = dict(PriorClaims=1, SuspiciousTiming=0, Docs=4.5, FollowUp=0, FraudRisk=0, Receipts=3.6, Witness=3.0)
p_maria = model.predict_proba(pd.DataFrame([maria])[features])[0, 1]

print(pd.DataFrame([maria], index=["Maria"]).to_string())
print(f"\nModel's probability that Maria's claim is high fraud risk: {p_maria:.3f}")
print("Decision:", "FLAGGED" if p_maria >= 0.5 else "fast track", "   |   Truth after full review: legitimate")

## Part 2: This quarter's closed claims

The analysts pull **6,000 claims from this quarter**, all of them fully reviewed, so each has a known outcome. Compared with the old history, each claim now has more recorded about it:

| Column | Meaning |
|---|---|
| ClaimID | An identifier |
| Area | The district where the loss occurred (Riverside or Northgate) |
| Shop | The repair shop that wrote the estimate |
| Adjuster | The adjuster who handled the claim |
| PriorClaims, SuspiciousTiming, Docs, FollowUp, Receipts, Witness | The same variables as before |
| FraudRisk | The outcome of the full review (1 = confirmed high fraud risk) |
| Note | The adjuster's free-text note |

The next cell produces the claims. **Do not worry about how yet.** We open the machine in Part 6. Treat the table as the insurer's records.

In [ ]:
# ----- The machine that produced this quarter's claims. We open it up in Part 6. -----
SHOPS = ["Apex Auto Body", "Bayview Collision", "Crescent Repair", "Delta Motors", "Eastside Auto"]
AREAS = ["Riverside", "Northgate"]
P_SHOP  = {"Riverside": [.06, .60, .12, .11, .11], "Northgate": [.06, .05, .30, .30, .29]}
P_PRIOR = {"Riverside": 0.40, "Northgate": 0.08}
RING_P  = 0.85

def draw_noise2(n, rng):
    return dict(area=rng.uniform(size=n), prior=rng.uniform(size=n), timing=rng.uniform(size=n),
                shop=rng.uniform(size=n), adj=rng.uniform(size=n), fraud=rng.uniform(size=n),
                docs=rng.normal(0, 1.5, n), follow=rng.uniform(size=n),
                receipts=rng.normal(0, 1.2, n), witness=rng.normal(0, 1.5, n))

def pick_shop(area, u):
    return np.array([SHOPS[int(np.searchsorted(np.cumsum(P_SHOP[a]), x))] for a, x in zip(area, u)])

def world2(U, do=None):
    do = do or {}
    d = {}
    d["Area"]             = do.get("Area", np.where(U["area"] < 0.25, "Riverside", "Northgate"))
    pp                    = np.where(d["Area"] == "Riverside", P_PRIOR["Riverside"], P_PRIOR["Northgate"])
    d["PriorClaims"]      = do.get("PriorClaims", (U["prior"] < pp).astype(int))
    d["SuspiciousTiming"] = do.get("SuspiciousTiming", (U["timing"] < 0.10).astype(int))
    d["Shop"]             = do.get("Shop", pick_shop(d["Area"], U["shop"]))
    d["Adjuster"]         = do.get("Adjuster", np.array([f"Adjuster {int(k) + 1}" for k in np.minimum((U["adj"] * 6).astype(int), 5)]))
    p = np.vectorize(lambda a, b: FRAUD_P[(int(a), int(b))])(d["PriorClaims"], d["SuspiciousTiming"])
    p = np.where(d["Shop"] == "Apex Auto Body", RING_P, p)
    d["FraudRisk"]        = do.get("FraudRisk", (U["fraud"] < p).astype(int))
    d["Docs"]             = do.get("Docs", 4 + 3.0 * d["PriorClaims"] + U["docs"])
    d["FollowUp"]         = do.get("FollowUp", (U["follow"] < 0.10).astype(int))
    d["Receipts"]         = do.get("Receipts", eq_receipts(d["FraudRisk"], d["Docs"], U["receipts"]))
    d["Witness"]          = do.get("Witness", eq_witness(d["FraudRisk"], d["FollowUp"], U["witness"]))
    return pd.DataFrame(d)

PHRASES = {
    "prior1":  ["claimant has filed before", "an earlier claim is on file"],
    "prior0":  ["no earlier claims on file", "this is the first claim"],
    "timing":  ["loss reported shortly after coverage increased", "reported soon after the policy limit was raised"],
    "rec_hi":  ["several receipts do not match the estimate", "multiple receipts disagree with the estimate"],
    "rec_mid": ["a few receipts need follow-up", "some receipts require follow-up"],
    "rec_lo":  ["receipts match the estimate", "receipts agree with the estimate"],
    "wit_hi":  ["witness statements conflict", "witness accounts disagree with each other"],
    "wit_mid": ["witness gave minor differences", "witness account differs slightly"],
    "wit_lo":  ["witness statements are consistent", "witness accounts agree"],
    "doc_hi":  ["documentation is thorough", "paperwork is complete"],
    "doc_lo":  ["documentation is thin", "paperwork is sparse"],
    "doc_mid": ["documentation is adequate", "paperwork is acceptable"],
}
def write_note(r, rng=None):
    pick = (lambda k: PHRASES[k][int(rng.integers(2))]) if rng is not None else (lambda k: PHRASES[k][0])
    parts = [pick("prior1" if r.PriorClaims == 1 else "prior0")]
    if r.SuspiciousTiming == 1: parts.append(pick("timing"))
    parts.append(pick("rec_hi" if r.Receipts >= 3 else "rec_mid" if r.Receipts >= 1.5 else "rec_lo"))
    parts.append(pick("wit_hi" if r.Witness >= 3 else "wit_mid" if r.Witness >= 1.5 else "wit_lo"))
    parts.append(pick("doc_hi" if r.Docs >= 7 else "doc_lo" if r.Docs < 5 else "doc_mid"))
    parts.append(f"repair estimate from {r.Shop}")
    parts.append(f"loss occurred in the {r.Area} district")
    return "; ".join(parts) + "."

N = 6000
claims = world2(draw_noise2(N, np.random.default_rng(14)))
claims.insert(0, "ClaimID", [f"C{i:04d}" for i in range(N)])
note_rng = np.random.default_rng(99)
claims["Note"] = [write_note(r, note_rng) for r in claims.itertuples()]

maria_ctx = dict(ClaimID="Maria", Area="Riverside", Shop="Bayview Collision", Adjuster="Adjuster 3", **maria)
maria_note = write_note(pd.Series(maria_ctx))
claims[["ClaimID", "Area", "Shop", "Adjuster", "FraudRisk", "Receipts", "Witness", "Note"]].head(5)

Each row is one closed claim, and the last column is the adjuster's note, written in ordinary words. Maria's own note reads:

In [ ]:
print(maria_note)

### How well does the model do on this quarter?
Before we dig in, check the model on these new claims.

In [ ]:
claims["Score"] = model.predict_proba(claims[features])[:, 1]
claims["Flagged"] = claims["Score"] >= 0.5
is_fraud = claims["FraudRisk"].values == 1
flag = claims["Flagged"].values

print(f"Share of this quarter's claims that are real fraud:  {is_fraud.mean():.1%}")
print(f"Share flagged by the model:                          {flag.mean():.1%}")
print(f"Share of real fraud the model catches (recall):      {flag[is_fraud].mean():.1%}")
print(f"Share of flags that are real fraud (precision):      {is_fraud[flag].mean():.1%}")
print(f"Share of legitimate claims flagged by error:         {flag[~is_fraud].mean():.1%}")

The model still works well. It misses a larger share of fraud than it did on the old history, though, and you will see why shortly.

### Each claim belongs to a customer (the CRM layer)
So far a claim is a row of clues. The insurer also knows the **customer** behind it, and keeps a **log of every contact** with that customer. These two records are the heart of CRM. We add two tables:

| Table | One row is | Columns |
|---|---|---|
| `customers` | One policyholder, linked to a claim | Tenure in years, annual premium, days to settle the claim, and whether the policy was renewed |
| `interactions` | One contact on a claim | The claim, what happened, and the day it happened (day 0 is the day the claim was filed) |

Maria's records are added by hand at the end, as her claim was. The cell that creates the numbers is another machine, and we open it in Part 7. Until then, treat the tables as the insurer's records.

In [ ]:
# ----- The CRM machine. We open it up in Part 7. -----
B0, B_TENURE, B_PRIOR, B_FLAG = 1.2, 0.10, -0.8, -0.9     # drivers of the chance of renewing
FLAG_WAIT_DAYS = 14                                         # a fraud review adds about two weeks

def sigmoid(x): return 1 / (1 + np.exp(-x))

def draw_crm_noise(n, rng):
    return dict(tenure=rng.gamma(2.0, 2.0, n), premium=rng.gamma(4.0, 120.0, n),
                wait=rng.gamma(4.0, 1.5, n), renew=rng.uniform(size=n))

def p_renew(prior, flagged, tenure):
    return sigmoid(B0 + B_TENURE * tenure + B_PRIOR * prior + B_FLAG * flagged)

def crm_outcomes(prior, flagged, V):
    tenure  = np.clip(V["tenure"], 0.5, 25)
    premium = 800 + V["premium"]
    wait    = V["wait"] + FLAG_WAIT_DAYS * flagged
    p       = p_renew(prior, flagged, tenure)
    renewed = (V["renew"] < p).astype(int)
    return tenure, premium, wait, p, renewed

def model_flag(df):
    return (model.predict_proba(df[features])[:, 1] >= 0.5).astype(int)

# ----- This quarter's customers -----
flag_q = claims["Flagged"].astype(int).values
tenure, premium, wait, p_ren, renewed = crm_outcomes(claims["PriorClaims"].values, flag_q, draw_crm_noise(N, np.random.default_rng(21)))

customers = pd.DataFrame(dict(CustomerID=[f"K{i:04d}" for i in range(N)], ClaimID=claims["ClaimID"].values,
                              Tenure=tenure.round(1), Premium=premium.round(0), WaitDays=wait.round(0).astype(int), Renewed=renewed))

def log_events(claim_id, flagged, wait_days, renewed):
    events = [("Claim filed", 0)]
    if flagged:
        events += [("Sent to fraud review", 3), ("Document request", 5)]
    events += [("Claim settled", wait_days), ("Renewed" if renewed else "Lapsed", wait_days + 45)]
    return [(claim_id, what, day) for what, day in events]

rows_ = []
for cid, f_, w_, r_ in zip(customers["ClaimID"], flag_q, customers["WaitDays"], customers["Renewed"]):
    rows_ += log_events(cid, f_, int(w_), r_)
interactions = pd.DataFrame(rows_, columns=["ClaimID", "Interaction", "Day"])

# ----- Maria, by hand: six years with us, a $1,350 premium, flagged, and she did not renew -----
maria_crm = dict(Tenure=6.0, Premium=1350.0, WaitDays=22, Renewed=0)
maria_events = [("Claim filed", 0), ("Sent to fraud review", 3), ("Document request", 5), ("Claim settled", 22), ("Lapsed", 67)]
customers = pd.concat([customers, pd.DataFrame([dict(CustomerID="Maria", ClaimID="Maria", **maria_crm)])], ignore_index=True)
interactions = pd.concat([interactions, pd.DataFrame([("Maria", w_, d_) for w_, d_ in maria_events], columns=interactions.columns)], ignore_index=True)

print(f"{len(customers):,} customers (this quarter's 6,000 plus Maria) and {len(interactions):,} logged interactions.\n")
print(customers.head(4).to_string(index=False))
print("\nMaria's customer record and her interaction log:")
print(customers[customers["ClaimID"] == "Maria"].to_string(index=False))
print(interactions[interactions["ClaimID"] == "Maria"].to_string(index=False))

Each flagged claim has two extra entries in its log (a trip to fraud review and a request for documents), and flagged claims take longer to settle. Keep an eye on that: the flag is an **action the insurer takes**, and it leaves a trace in the customer's record.

## Part 3: Represent and retrieve: who reads like Maria? (rung 1)

### From words to numbers
A computer cannot compare two notes as sentences, so we turn each note into a list of numbers. This is a **representation**. The simplest useful one you have seen before is **TF-IDF**:

1. **Count** how many times each word appears in a note (term frequency).
2. **Down-weight common words.** A word that appears in almost every note tells us little, so we multiply by a weight that is **small for common words and large for rare ones** (inverse document frequency).
3. **Scale** each note so its list of numbers has length 1. Then long and short notes can be compared fairly.

**Similarity** between two notes is the **cosine** of the angle between their lists: 1 means pointing the same way (the same words), 0 means nothing in common. When both lists are scaled to length 1, cosine is just the dot product. We build all of it by hand.

In [ ]:
STOP = {"the", "a", "in", "to", "from", "of", "on", "is", "are", "this"}
def tokens(text): return [w for w in re.findall(r"[a-z]+", text.lower()) if w not in STOP]

docs_tok = [tokens(t) for t in claims["Note"]]
vocab = {w: i for i, w in enumerate(sorted({w for d in docs_tok for w in d}))}

doc_freq = np.zeros(len(vocab))
for d in docs_tok:
    for w in set(d):
        doc_freq[vocab[w]] += 1
idf = np.log((1 + N) / (1 + doc_freq)) + 1               # small for common words, large for rare ones

def to_vector(toks):
    v = np.zeros(len(vocab))
    for w in toks:
        if w in vocab: v[vocab[w]] += 1                   # 1. count
    v = v * idf                                           # 2. weight by rarity
    n = np.linalg.norm(v)
    return v / n if n else v                              # 3. scale to length 1

X = np.array([to_vector(d) for d in docs_tok])
print(f"{len(vocab)} distinct words; each note is now a list of {X.shape[1]} numbers.")
rare = sorted(vocab, key=lambda w: -idf[vocab[w]])[:4]
common = sorted(vocab, key=lambda w: idf[vocab[w]])[:4]
print("Rarest words (highest weight):   ", ", ".join(f"{w} ({idf[vocab[w]]:.2f})" for w in rare))
print("Most common words (lowest weight):", ", ".join(f"{w} ({idf[vocab[w]]:.2f})" for w in common))

**A check against a library.** The same recipe is built into scikit-learn. If we did it correctly, the two should agree to rounding error.

In [ ]:
tv = TfidfVectorizer(tokenizer=tokens, lowercase=False, token_pattern=None).fit(claims["Note"])
sk_sims = (tv.transform(claims["Note"]) @ tv.transform([maria_note]).T).toarray().ravel()

q = to_vector(tokens(maria_note))
sims = X @ q                                              # dot product = cosine, because every vector has length 1
print(f"Largest difference from scikit-learn: {np.abs(sk_sims - sims).max():.1e}")

### Retrieve: the claims most like Maria's
**Retrieval** is just ranking by similarity and taking the top few. This is the "R" in RAG: given a question (here, Maria's note), fetch the most relevant past records.

In [ ]:
order = np.argsort(-sims)
top = claims.iloc[order[:25]]
look = claims.iloc[order[:5]][["ClaimID", "FraudRisk", "Shop", "Note"]].assign(Similarity=sims[order[:5]])
print("The five notes most similar to Maria's:\n")
print(look.to_string(index=False))

near = claims[sims >= 0.9]
print(f"\nClaims with a note at least 0.9 similar to hers: {len(near)}, of which fraud: {near['FraudRisk'].mean():.0%}")
print(f"Among the 25 most similar claims, share that were fraud: {top['FraudRisk'].mean():.0%}")
print(f"Share of all claims this quarter that were fraud:       {claims['FraudRisk'].mean():.0%}")

**Reading the result.** Claims that read like Maria's were almost all fraud. That is what the model said too, for the same reason: her note has the words of a fraudulent claim (receipts that do not match, witnesses who disagree, a prior claim). Retrieval gave us **context**, and the context points the wrong way for Maria.

This is the lesson of rung 1: **a look-alike is not a cause, and it is not the same case.** The honest claims that share her symptoms are exactly the false alarms, and similarity alone cannot tell her apart from the fraudulent claims around her.

### A note on a better representation
TF-IDF matches **words**. Two notes that say the same thing in different words ("several receipts do not match" and "multiple receipts disagree") share few words, so TF-IDF scores them as only partly alike. A **sentence embedding** model turns each note into numbers meant to capture meaning, so paraphrases land close together. The retrieval step stays exactly the same (rank by cosine, take the top few); only the representation changes. We stay with TF-IDF here because you can see every number in it.

### Keep the vectors in a database (ChromaDB)
So far the vectors live in memory and vanish when the notebook closes. A **vector database** stores them on disk and answers one question: *which stored vectors are closest to this one?* That is the retrieval step as a service, and it is the storage layer behind RAG. **ChromaDB** keeps its data in a folder inside this project (`data/chroma`), so the collection is still there the next time you open the notebook.

We store our own TF-IDF vectors (so Chroma does not need to download an embedding model) along with each claim's note and a little metadata, then ask for Maria's nearest neighbors. This matches the hand-built ranking, which is the point: the database does the same job at scale.

In [ ]:
import chromadb

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
CHROMA_DIR = ROOT / "data" / "chroma"
chroma = chromadb.PersistentClient(path=str(CHROMA_DIR))

try:
    chroma.delete_collection("claim_notes")                  # start fresh so the cell is safe to re-run
except Exception:
    pass
notes_col = chroma.create_collection("claim_notes", metadata={"hnsw:space": "cosine"})

for i in range(0, N, 2000):
    part = claims.iloc[i:i + 2000]
    notes_col.add(ids=part["ClaimID"].tolist(), embeddings=X[i:i + 2000].tolist(), documents=part["Note"].tolist(),
                  metadatas=[{"fraud": int(f), "shop": sh} for f, sh in zip(part["FraudRisk"], part["Shop"])])

hits = notes_col.query(query_embeddings=[q.tolist()], n_results=25, include=["metadatas", "distances"])
chroma_share = np.mean([m["fraud"] for m in hits["metadatas"][0]])
overlap = len(set(hits["ids"][0]) & set(top["ClaimID"]))
print(f"Stored {notes_col.count():,} notes in {CHROMA_DIR}")
print(f"ChromaDB's 25 nearest notes to Maria's: {100 * chroma_share:.0f}% fraud   (hand-built ranking: {100 * top['FraudRisk'].mean():.0f}%)")
print(f"Claims that appear in both top-25 lists: {overlap} of 25   (they should agree closely; many notes are word-for-word equal, so ties can be broken differently)")

In [ ]:
# Persistence check: open the folder again with a brand-new client, as if the notebook had been restarted
reopened = chromadb.PersistentClient(path=str(CHROMA_DIR)).get_collection("claim_notes")
print(f"After reopening, the collection still holds {reopened.count():,} notes.")

## Part 4: Cluster: which group does Maria fall into? (rung 1)

Retrieval answers "who is *most* like her?" **Clustering** answers a different question: "if we sort *all* claims into a few groups of similar claims, which group is hers?" Nobody tells the algorithm what the groups are. It finds them from the numbers.

### k-means, by hand
The idea is simple:
1. Pick *k* starting centers.
2. **Assign** every claim to its nearest center.
3. **Move** each center to the average of the claims assigned to it.
4. Repeat steps 2 and 3 until nothing moves.

We describe each claim with five numbers (prior claims, suspicious timing, receipts, witness, documents), each standardized so that no single number dominates the distance.

In [ ]:
cols = ["PriorClaims", "SuspiciousTiming", "Receipts", "Witness", "Docs"]
mu, sd = claims[cols].mean(), claims[cols].std()
Z = ((claims[cols] - mu) / sd).values

def kmeans(Z, k, seed, iters=100):
    rng = np.random.default_rng(seed)
    centers = [Z[rng.integers(len(Z))]]
    for _ in range(k - 1):                                  # spread the starting centers out
        d2 = np.min([((Z - c) ** 2).sum(1) for c in centers], axis=0)
        centers.append(Z[rng.choice(len(Z), p=d2 / d2.sum())])
    centers = np.array(centers)
    for _ in range(iters):
        label = np.argmin(((Z[:, None, :] - centers[None]) ** 2).sum(2), axis=1)              # assign
        new = np.array([Z[label == j].mean(0) if (label == j).any() else centers[j] for j in range(k)])   # move
        if np.allclose(new, centers): break
        centers = new
    return label, centers, ((Z - centers[label]) ** 2).sum()

label, centers, inertia = min((kmeans(Z, 4, s) for s in range(10)), key=lambda t: t[2])   # best of 10 starts
claims["Cluster"] = label

summary = claims.groupby("Cluster").agg(
    claims=("ClaimID", "size"), fraud_share=("FraudRisk", "mean"),
    prior=("PriorClaims", "mean"), timing=("SuspiciousTiming", "mean"),
    receipts=("Receipts", "mean"), witness=("Witness", "mean"), docs=("Docs", "mean"))
z_maria = ((pd.Series(maria_ctx)[cols].astype(float) - mu) / sd).values
maria_cluster = int(np.argmin(((centers - z_maria) ** 2).sum(1)))
summary["maria_here"] = ["<-- Maria" if c == maria_cluster else "" for c in summary.index]
summary

**Reading the groups.** Each row is a group the algorithm found. Look at the averages and describe each group in your own words:
- One group is nearly all **legitimate claims with few warning signs**, and it holds the largest share of claims.
- One is **claimants with prior claims**, and about half of them are fraud. **Maria lands here.**
- One is made of claims with **suspicious timing**, with a similar fraud share.
- One has **high receipts mismatch but very few documents**, and it is mostly legitimate. These are people whose paperwork is thin, not people who are padding claims. That is the lever from earlier showing up as a group.

A picture helps. We squeeze the five numbers into two with a hand-built projection (the directions along which claims vary most) and color each claim by its real outcome.

In [ ]:
_, _, Vt = np.linalg.svd(Z, full_matrices=False)
P = Z @ Vt[:2].T
pm = z_maria @ Vt[:2].T

fig, ax = plt.subplots(figsize=(7.6, 5.4))
fraud_mask = claims["FraudRisk"].values == 1
ax.scatter(P[~fraud_mask, 0], P[~fraud_mask, 1], s=6, alpha=0.30, color="#4a90c2", label="legitimate")
ax.scatter(P[fraud_mask, 0], P[fraud_mask, 1], s=6, alpha=0.35, color="#d9534f", label="fraud")
cp = centers @ Vt[:2].T
for j in range(len(cp)):
    ax.text(cp[j, 0], cp[j, 1], str(j), fontsize=13, weight="bold", ha="center", va="center",
            bbox=dict(boxstyle="circle", fc="white", ec="#333"))
ax.scatter(*pm, marker="*", s=380, color="gold", edgecolor="black", zorder=5, label="Maria")
ax.set_xlabel("first direction"); ax.set_ylabel("second direction")
ax.set_title("Clusters of claims (numbers mark the centers)")
from matplotlib.lines import Line2D
handles = [Line2D([], [], marker="o", ls="", color="#4a90c2", markersize=6, label="legitimate"),
           Line2D([], [], marker="o", ls="", color="#d9534f", markersize=6, label="fraud"),
           Line2D([], [], marker="*", ls="", color="gold", markeredgecolor="black", markersize=14, label="Maria")]
ax.legend(handles=handles); plt.tight_layout(); plt.show()

A note on the picture: it squeezes five numbers into two, so groups overlap, and Maria's star appears between the prior-claims group and the suspicious-timing group. In all five numbers she is closest to the center of the prior-claims group, which is where the table puts her.

**What clustering can and cannot say.** Maria's group is about half fraud, so a group label alone would send her to review, and it would be wrong for her. Clusters describe **what claims look like**. They do not say **why** a claim is in a group, or what would happen if a claim changed. They are a rung 1 summary, as useful and as limited as the look-alike search.

### The same groups, read as customer segments
In CRM, a cluster of similar customers is a **segment**: a group that can be treated differently. We attach the customer records to the clusters and read the groups as customers instead of claims.

In [ ]:
cust = (claims[["ClaimID", "Cluster", "Shop", "Area", "FraudRisk"]]
        .assign(Flagged=claims["Flagged"].astype(int).values)
        .merge(customers, on="ClaimID"))

segments = cust.groupby("Cluster").agg(
    customers=("ClaimID", "size"),
    flagged_pct=("Flagged", lambda s: 100 * s.mean()),
    avg_tenure_yrs=("Tenure", "mean"),
    avg_premium=("Premium", "mean"),
    avg_days_to_settle=("WaitDays", "mean"),
    renewed_pct=("Renewed", lambda s: 100 * s.mean()))
segments["maria_here"] = ["<-- Maria" if c == maria_cluster else "" for c in segments.index]
segments

**Reading the segments.** Tenure and premium look much the same in every group, so they do not separate customers here. Two columns do. The groups where many claims are flagged **wait longer for their claim to settle**, and they **renew less often**. Maria's segment is one of them.

This is a **rung 1** finding, and it is useful for planning: it tells the insurer which customers to look after. It does not say that the flag *causes* the lower renewal. The people in the flagged groups also differ in other ways (they have prior claims, for example), and sorting out which difference does the damage is a rung 2 question. We come back to it in Part 7.

## Part 5: The knowledge graph: Maria's network (rung 1)

So far each claim has stood alone. But claims are connected: they share **shops**, **areas** and **adjusters**. A **knowledge graph** records those connections directly, as nodes and edges, so we can ask questions that follow them.

| Node type | Examples |
|---|---|
| Claim | C0000, C0001, ..., Maria |
| Shop | Apex Auto Body, Bayview Collision, ... |
| Area | Riverside, Northgate |
| Adjuster | Adjuster 1, ..., Adjuster 6 |

| Edge | Meaning |
|---|---|
| Claim → Shop (`REPAIRED_AT`) | The shop that wrote the estimate |
| Claim → Area (`IN_AREA`) | Where the loss occurred |
| Claim → Adjuster (`HANDLED_BY`) | Who handled the claim |

### Build it in NetworkX (hand-built)

In [ ]:
G = nx.Graph()
for r in claims.itertuples():
    G.add_node(r.ClaimID, kind="Claim", fraud=int(r.FraudRisk))
    G.add_edge(r.ClaimID, r.Shop, rel="REPAIRED_AT")
    G.add_edge(r.ClaimID, r.Area, rel="IN_AREA")
    G.add_edge(r.ClaimID, r.Adjuster, rel="HANDLED_BY")
for n in SHOPS: G.nodes[n]["kind"] = "Shop"
for n in AREAS: G.nodes[n]["kind"] = "Area"
for n in sorted(claims["Adjuster"].unique()): G.nodes[n]["kind"] = "Adjuster"

kinds = pd.Series({n: d["kind"] for n, d in G.nodes(data=True)}).value_counts()
print(f"{G.number_of_nodes():,} nodes and {G.number_of_edges():,} edges")
print(kinds.to_string())

### Ask it a question: how risky is each shop?
In a graph, "claims at a shop" means **follow the edges** from the shop node to its claim neighbors.

In [ ]:
def neighbors_of_kind(graph, node, kind):
    return [n for n in graph[node] if graph.nodes[n]["kind"] == kind]

def group_table(graph, kind):
    rows = []
    for node in [n for n, d in graph.nodes(data=True) if d["kind"] == kind]:
        cl = neighbors_of_kind(graph, node, "Claim")
        rows.append((node, len(cl), 100 * np.mean([graph.nodes[c]["fraud"] for c in cl])))
    return pd.DataFrame(rows, columns=[kind.lower(), "claims", "fraud_pct"]).sort_values("fraud_pct", ascending=False).reset_index(drop=True)

shop_table = group_table(G, "Shop")
shop_table

**Two shops stand out.** **Apex Auto Body** has a very high share of fraudulent claims. **Bayview Collision** is also well above the others, and it is the shop on Maria's claim. Compare the areas and the adjusters:

In [ ]:
print(group_table(G, "Area").to_string(index=False))
print()
print(group_table(G, "Adjuster").to_string(index=False))

Riverside has about twice the fraud share of Northgate. The six adjusters all sit in a narrow band. Their small differences are what chance alone produces, so the adjusters give us nothing to act on. A node can be well connected without being risky.

### The same graph in Neo4j
NetworkX holds the graph in memory, and it disappears when the notebook closes. A **graph database** like Neo4j stores it, and lets you ask questions in a query language, **Cypher**. A Cypher query draws the pattern you want with parentheses for nodes and arrows for edges:

```
(c:Claim)-[:REPAIRED_AT]->(s:Shop)       a claim connected to a shop
```

The next cell loads this quarter's claims into your local Neo4j. It also loads the customers and their interaction logs. **It deletes only nodes labeled Claim, Shop, Area, Adjuster, Customer or Interaction**, so other graphs in the same database (such as the Bob and Alice demo) are left alone.

In [ ]:
CREATE_CLAIMS = '''
UNWIND $rows AS row
CREATE (:Claim {id: row.id, fraud: row.fraud, prior: row.prior, timing: row.timing, docs: row.docs,
                receipts: row.receipts, witness: row.witness, flagged: row.flagged, score: row.score,
                shop: row.shop, area: row.area, adjuster: row.adjuster})
'''
LINK_QUERIES = [
    "MATCH (c:Claim) MERGE (s:Shop {name: c.shop}) MERGE (c)-[:REPAIRED_AT]->(s)",
    "MATCH (c:Claim) MERGE (a:Area {name: c.area}) MERGE (c)-[:IN_AREA]->(a)",
    "MATCH (c:Claim) MERGE (j:Adjuster {name: c.adjuster}) MERGE (c)-[:HANDLED_BY]->(j)",
    "MATCH (c:Claim) REMOVE c.shop, c.area, c.adjuster",
]

if NEO4J_OK:
    run_cypher("MATCH (n) WHERE n:Claim OR n:Shop OR n:Area OR n:Adjuster OR n:Customer OR n:Interaction DETACH DELETE n")
    run_cypher("CREATE CONSTRAINT claim_id IF NOT EXISTS FOR (c:Claim) REQUIRE c.id IS UNIQUE")
    neo = claims.rename(columns={"ClaimID": "id", "FraudRisk": "fraud", "PriorClaims": "prior", "SuspiciousTiming": "timing",
                                 "Docs": "docs", "Receipts": "receipts", "Witness": "witness",
                                 "Shop": "shop", "Area": "area", "Adjuster": "adjuster",
                                 "Flagged": "flagged", "Score": "score"})
    neo["flagged"] = neo["flagged"].astype(int)
    neo = neo[["id", "fraud", "prior", "timing", "docs", "receipts", "witness", "flagged", "score", "shop", "area", "adjuster"]]
    rows = json.loads(neo.to_json(orient="records"))
    for i in range(0, len(rows), 1000):
        run_cypher(CREATE_CLAIMS, rows=rows[i:i + 1000])
    for q_ in LINK_QUERIES:
        run_cypher(q_)
    counts = run_cypher("MATCH (n) WHERE n:Claim OR n:Shop OR n:Area OR n:Adjuster RETURN labels(n)[0] AS label, count(n) AS nodes ORDER BY label")
    print(pd.DataFrame(counts).to_string(index=False))
else:
    print("Neo4j not connected: skipped. Nothing was loaded.")

**Customers and interactions go in as nodes too.** Each customer is linked to the claim they filed, and each interaction is linked to its claim. That is what makes the graph a relationship map: from one customer you can walk to their claim, their shop, their area, their adjuster and every contact we have had with them.

| New node | Edge |
|---|---|
| Customer | `(:Customer)-[:FILED]->(:Claim)` |
| Interaction | `(:Claim)-[:HAS_INTERACTION]->(:Interaction)` |

In [ ]:
CREATE_CUSTOMERS = '''
UNWIND $rows AS row
MATCH (c:Claim {id: row.claim})
CREATE (cu:Customer {id: row.id, tenure: row.tenure, premium: row.premium, renewed: row.renewed})
CREATE (cu)-[:FILED]->(c)
'''
CREATE_INTERACTIONS = '''
UNWIND $rows AS row
MATCH (c:Claim {id: row.claim})
CREATE (c)-[:HAS_INTERACTION]->(:Interaction {type: row.type, day: row.day})
'''

if NEO4J_OK:
    quarter = customers[customers["ClaimID"] != "Maria"]
    cust_rows = json.loads(quarter.rename(columns={"CustomerID": "id", "ClaimID": "claim", "Tenure": "tenure",
                                                   "Premium": "premium", "Renewed": "renewed"})
                           [["id", "claim", "tenure", "premium", "renewed"]].to_json(orient="records"))
    for i in range(0, len(cust_rows), 1000):
        run_cypher(CREATE_CUSTOMERS, rows=cust_rows[i:i + 1000])
    int_rows = json.loads(interactions[interactions["ClaimID"] != "Maria"]
                          .rename(columns={"ClaimID": "claim", "Interaction": "type", "Day": "day"}).to_json(orient="records"))
    for i in range(0, len(int_rows), 5000):
        run_cypher(CREATE_INTERACTIONS, rows=int_rows[i:i + 5000])
    counts = run_cypher("MATCH (n) WHERE n:Claim OR n:Customer OR n:Interaction RETURN labels(n)[0] AS label, count(n) AS nodes ORDER BY label")
    print(pd.DataFrame(counts).to_string(index=False))
else:
    print("Neo4j not connected: skipped. Nothing was loaded.")

**The same question in Cypher: how risky is each shop?** Read it like a sentence: *match every claim that was repaired at a shop; for each shop, return the number of claims and the fraud share.*

In [ ]:
CYPHER_SHOPS = '''
MATCH (c:Claim)-[:REPAIRED_AT]->(s:Shop)
RETURN s.name AS shop, count(c) AS claims, round(100.0 * avg(c.fraud), 1) AS fraud_pct
ORDER BY fraud_pct DESC
'''
neo_shops = show_cypher(CYPHER_SHOPS)
neo_shops

In [ ]:
if NEO4J_OK:
    same = (neo_shops.sort_values("shop").reset_index(drop=True)["claims"].tolist()
            == shop_table.sort_values("shop").reset_index(drop=True)["claims"].tolist())
    print("Neo4j and NetworkX agree on the claim counts:", same)

**A pattern across two hops: shop and area together.** This one follows two edges from each claim.

In [ ]:
CYPHER_SHOP_AREA = '''
MATCH (c:Claim)-[:IN_AREA]->(a:Area), (c)-[:REPAIRED_AT]->(s:Shop)
RETURN a.name AS area, s.name AS shop, count(c) AS claims, round(100.0 * avg(c.fraud), 1) AS fraud_pct
ORDER BY area, shop
'''
show_cypher(CYPHER_SHOP_AREA)

In [ ]:
shop_area = (claims.groupby(["Area", "Shop"]).agg(claims=("ClaimID", "size"), fraud_pct=("FraudRisk", lambda s: 100 * s.mean()))
             .reset_index())
shop_area

### Add Maria to the graph and look at her neighborhood
Maria's claim joins the graph as one more node, connected to her shop, her area and her adjuster. Then we ask for the **facts** about her claim, the same way the earlier graph RAG demo retrieved facts about a person.

In [ ]:
G.add_node("Maria", kind="Claim", fraud=0)
G.add_edge("Maria", "Bayview Collision", rel="REPAIRED_AT")
G.add_edge("Maria", "Riverside", rel="IN_AREA")
G.add_edge("Maria", "Adjuster 3", rel="HANDLED_BY")

maria_facts = [("Maria", G.edges["Maria", n]["rel"], n) for n in G["Maria"]]
print("Facts about Maria's claim (NetworkX):")
for s_, p_, o_ in maria_facts: print(f"  {s_} -{p_}-> {o_}")

peers = [c for c in neighbors_of_kind(G, "Bayview Collision", "Claim") if c != "Maria"]
peer_fraud = 100 * np.mean([G.nodes[c]["fraud"] for c in peers])
print(f"\nOther claims at Bayview Collision: {len(peers):,}, of which {peer_fraud:.1f}% were fraud")

In [ ]:
ADD_MARIA = '''
MATCH (s:Shop {name: $shop}), (a:Area {name: $area}), (j:Adjuster {name: $adjuster})
CREATE (m:Claim {id: 'Maria', fraud: 0, prior: 1, timing: 0, docs: 4.5, receipts: 3.6, witness: 3.0, flagged: 1, score: $score})
CREATE (m)-[:REPAIRED_AT]->(s)
CREATE (m)-[:IN_AREA]->(a)
CREATE (m)-[:HANDLED_BY]->(j)
'''
CYPHER_MARIA_FACTS = '''
MATCH (m:Claim {id: 'Maria'})-[r]->(x)
WHERE x:Shop OR x:Area OR x:Adjuster
RETURN type(r) AS predicate, x.name AS object
'''
CYPHER_MARIA_PEERS = '''
MATCH (m:Claim {id: 'Maria'})-[:REPAIRED_AT]->(s:Shop)<-[:REPAIRED_AT]-(o:Claim)
WHERE o.id <> 'Maria'
RETURN s.name AS shop, count(o) AS peers, round(100.0 * avg(o.fraud), 1) AS fraud_pct
'''
if NEO4J_OK:
    run_cypher("MATCH (:Claim {id: 'Maria'})-[:HAS_INTERACTION]->(i:Interaction) DETACH DELETE i")   # safe to re-run
    run_cypher("MATCH (cu:Customer {id: 'Maria'}) DETACH DELETE cu")
    run_cypher("MATCH (m:Claim {id: 'Maria'}) DETACH DELETE m")
    run_cypher(ADD_MARIA, shop="Bayview Collision", area="Riverside", adjuster="Adjuster 3", score=float(p_maria))
show_cypher(CYPHER_MARIA_FACTS)

In [ ]:
show_cypher(CYPHER_MARIA_PEERS)

### Customer 360: everything about Maria in one place
A **customer 360** is one view of everything the insurer knows about one customer: who she is, what she filed, who handled it, and every contact since. In a spreadsheet world that means stitching several files together. In a graph it is one walk outward from the customer. First the same view built from our tables, then the Cypher version.

In [ ]:
def customer_360(claim_id):
    cu = customers[customers["ClaimID"] == claim_id].iloc[0]
    if claim_id == "Maria":
        info = dict(Shop="Bayview Collision", Area="Riverside", Adjuster="Adjuster 3", PriorClaims=1,
                    Score=p_maria, Cluster=maria_cluster)
    else:
        info = claims[claims["ClaimID"] == claim_id].iloc[0].to_dict()
        info["Cluster"] = int(cust.loc[cust["ClaimID"] == claim_id, "Cluster"].iloc[0])
    seg = cust[cust["Cluster"] == info["Cluster"]]
    shop_peers = cust[(cust["Shop"] == info["Shop"]) & (cust["ClaimID"] != claim_id)]
    log = interactions[interactions["ClaimID"] == claim_id].sort_values("Day")
    print(f"CUSTOMER 360: {claim_id}")
    print("-" * 70)
    print(f"Profile:   {cu.Tenure:.1f} years with us, annual premium ${cu.Premium:,.0f}, {int(info['PriorClaims'])} earlier claim(s)")
    print(f"Claim:     repaired at {info['Shop']}, loss in {info['Area']}, handled by {info['Adjuster']}, model score {info['Score']:.3f}")
    print(f"Segment:   group {info['Cluster']}, {len(seg):,} customers, {100 * seg['Renewed'].mean():.0f}% of them renewed")
    print(f"Shop:      {len(shop_peers):,} other {info['Shop']} customers, {100 * shop_peers['Renewed'].mean():.0f}% renewed")
    print("Timeline:  " + "  ->  ".join(f"day {int(d)}: {w}" for w, d in zip(log["Interaction"], log["Day"])))

customer_360("Maria")

In [ ]:
ADD_MARIA_CRM = '''
MATCH (m:Claim {id: 'Maria'})
CREATE (cu:Customer {id: 'Maria', tenure: $tenure, premium: $premium, renewed: $renewed})
CREATE (cu)-[:FILED]->(m)
WITH m
UNWIND $events AS e
CREATE (m)-[:HAS_INTERACTION]->(:Interaction {type: e.type, day: e.day})
'''
CYPHER_360_PROFILE = '''
MATCH (cu:Customer)-[:FILED]->(m:Claim {id: 'Maria'})-[:REPAIRED_AT]->(s:Shop),
      (m)-[:IN_AREA]->(a:Area), (m)-[:HANDLED_BY]->(j:Adjuster)
RETURN cu.tenure AS tenure_years, cu.premium AS premium, cu.renewed AS renewed,
       m.flagged AS flagged, round(m.score, 3) AS score, s.name AS shop, a.name AS area, j.name AS adjuster
'''
CYPHER_360_TIMELINE = '''
MATCH (:Claim {id: 'Maria'})-[:HAS_INTERACTION]->(i:Interaction)
RETURN i.day AS day, i.type AS interaction
ORDER BY day
'''
if NEO4J_OK:
    run_cypher(ADD_MARIA_CRM, tenure=maria_crm["Tenure"], premium=maria_crm["Premium"], renewed=maria_crm["Renewed"],
               events=[{"type": w_, "day": d_} for w_, d_ in maria_events])
show_cypher(CYPHER_360_PROFILE)

In [ ]:
show_cypher(CYPHER_360_TIMELINE)

**A retention question in Cypher.** Among customers whose claim was **legitimate**, compare renewal for those who were flagged with those who were not. This is a rung 1 comparison, read straight from the graph.

In [ ]:
CYPHER_RETENTION = '''
MATCH (cu:Customer)-[:FILED]->(c:Claim)
WHERE c.fraud = 0 AND c.id <> 'Maria'
RETURN c.flagged AS flagged, count(cu) AS customers, round(100.0 * avg(cu.renewed), 1) AS renewed_pct
ORDER BY flagged
'''
neo_ret = show_cypher(CYPHER_RETENTION)
neo_ret

In [ ]:
honest_q = cust[cust["FraudRisk"] == 0]
(honest_q.groupby("Flagged").agg(customers=("ClaimID", "size"), renewed_pct=("Renewed", lambda s: 100 * s.mean()))
         .reset_index())

Honest customers who were flagged renewed less often than honest customers who were not. How much of that gap is *caused by the flag*? The graph cannot say. We answer that in Part 7.

### The picture
Each shop is placed next to the area its claims come from. A thicker line means more claims, and darker red means a higher fraud share.

In [ ]:
import matplotlib.colors as mcolors
cm = plt.cm.Reds
fig, ax = plt.subplots(figsize=(9.5, 5.6))
area_pos = {"Riverside": (0, 1.2), "Northgate": (0, -1.2)}
shop_pos = {s: (4, y) for s, y in zip(SHOPS, [2.4, 1.2, 0, -1.2, -2.4])}
share = shop_table.set_index("shop")["fraud_pct"]
cnt = shop_area.set_index(["Area", "Shop"])["claims"]
for (a, s), n in cnt.items():
    ax.plot([area_pos[a][0], shop_pos[s][0]], [area_pos[a][1], shop_pos[s][1]], color="#888", lw=0.4 + n / 160, alpha=0.55, zorder=1)
for s, (x, y) in shop_pos.items():
    ax.scatter(x, y, s=2600, color=cm(share[s] / 100 * 1.15), edgecolor="black" if s == "Bayview Collision" else "#777",
               linewidth=3 if s == "Bayview Collision" else 1, zorder=2)
    ax.text(x + 0.45, y, f"{s}\n{share[s]:.0f}% fraud", va="center", fontsize=9)
for a, (x, y) in area_pos.items():
    ax.scatter(x, y, s=2600, color="#c8c8c8", edgecolor="#777", zorder=2)
    ax.text(x - 0.45, y, a, va="center", ha="right", fontsize=9)
mx, my = 2.0, 3.2
ax.scatter(mx, my, marker="*", s=420, color="gold", edgecolor="black", zorder=4)
ax.text(mx, my + 0.35, "Maria", ha="center", fontsize=10, weight="bold")
ax.plot([mx, area_pos["Riverside"][0]], [my, area_pos["Riverside"][1]], color="gold", lw=2, zorder=3)
ax.plot([mx, shop_pos["Bayview Collision"][0]], [my, shop_pos["Bayview Collision"][1]], color="gold", lw=2, zorder=3)
ax.set_xlim(-2.4, 7.4); ax.set_ylim(-3.3, 4.2); ax.axis("off")
ax.set_title("Areas, shops and Maria: line thickness = number of claims, shop color = fraud share")
plt.tight_layout(); plt.show()

### What the graph says, and what we are tempted to conclude
Two shops stand out, and Maria is attached to one of them. It is tempting to write two **hypotheses** straight from the picture:

- **H1:** Claims at **Bayview Collision** are fraud-prone *because of the shop*.
- **H2:** Claims at **Apex Auto Body** are fraud-prone *because of the shop*.

Both are rung 1 patterns so far: a shop and fraud go together. Nothing here says that **changing** what happens at either shop would change the fraud. That is a rung 2 question, and we test it next.

## Part 6: Validate causally: does the shop matter? (rung 2)

To test "because of the shop", we need a graph of what we believe causes what. Here is one, drawn from how these claims arise:
- **Area** affects who has prior claims. Some districts have more claimants with a history, and **Area also affects which shop people use**: people go to nearby shops.
- **PriorClaims** and **SuspiciousTiming** affect fraud, as before.
- **Shop** *might* affect fraud. That arrow is the hypothesis we want to test.
- **FraudRisk** produces the clues the model reads.

Look at the **orange path**. It runs from Shop back to Area, then through PriorClaims to FraudRisk. It carries association, and none of it is an effect of the shop. This is a **backdoor path**, and Area is the common cause behind it.

In [ ]:
cg = nx.DiGraph([("Area", "Shop"), ("Area", "PriorClaims"), ("PriorClaims", "FraudRisk"), ("SuspiciousTiming", "FraudRisk"),
                 ("Shop", "FraudRisk"), ("FraudRisk", "Receipts"), ("FraudRisk", "Witness")])
cpos = {"Area": (0, 1.6), "Shop": (0, -0.4), "PriorClaims": (2.6, 1.6), "SuspiciousTiming": (2.6, -1.4),
        "FraudRisk": (5.2, 0.4), "Receipts": (7.8, 1.2), "Witness": (7.8, -0.4)}
ccolors = {"Area": "#c8c8c8", "Shop": "#b2df8a", "PriorClaims": "#c8c8c8", "SuspiciousTiming": "#c8c8c8",
           "FraudRisk": "#fb9a99", "Receipts": "#fdbf6f", "Witness": "#fdbf6f"}
backdoor = {("Area", "Shop"), ("Area", "PriorClaims"), ("PriorClaims", "FraudRisk")}
fig, ax = plt.subplots(figsize=(10.5, 4.8))
nx.draw_networkx_nodes(cg, cpos, ax=ax, node_size=4300, node_color=[ccolors[n] for n in cg.nodes])
nx.draw_networkx_labels(cg, cpos, ax=ax, font_size=8)
nx.draw_networkx_edges(cg, cpos, ax=ax, edgelist=[e for e in cg.edges if e not in backdoor], node_size=4300,
                       arrowsize=22, edge_color="#444", width=1.6)
nx.draw_networkx_edges(cg, cpos, ax=ax, edgelist=[e for e in cg.edges if e in backdoor], node_size=4300,
                       arrowsize=22, edge_color="#e08a00", width=3)
ax.text(1.3, 2.25, "Backdoor path:  Shop <- Area -> PriorClaims -> FraudRisk", color="#e08a00", fontsize=9, ha="center")
ax.set_xlim(-1.2, 8.8); ax.set_ylim(-2.1, 2.7); ax.axis("off")
ax.set_title("Hypothesized causal graph. Orange = the backdoor path from Shop to FraudRisk.")
plt.show()

### Step 1: Look at the data
Compare the fraud share of claims at the shop with claims at the three shops that raised no flags (Crescent Repair, Delta Motors, Eastside Auto).

In [ ]:
REFERENCE = ["Crescent Repair", "Delta Motors", "Eastside Auto"]
def naive_effect(shop):
    sub = claims[claims["Shop"].isin([shop] + REFERENCE)]
    return sub[sub["Shop"] == shop]["FraudRisk"].mean() - sub[sub["Shop"].isin(REFERENCE)]["FraudRisk"].mean()

print("SEEING (the data alone): extra fraud share at the shop compared with the three reference shops")
print(f"  Bayview Collision: {100 * naive_effect('Bayview Collision'):+.1f} percentage points")
print(f"  Apex Auto Body:    {100 * naive_effect('Apex Auto Body'):+.1f} percentage points")

Both look bad, and Bayview Collision looks like a shop with a problem. That is the rung 1 reading.

### Step 2: Hold the common cause fixed
To block the backdoor path, compare shops **within the same area** and then combine the areas, weighting by how common each area is. This is the same adjustment as before, with Area as the thing held fixed.

In [ ]:
area_share = claims["Area"].value_counts(normalize=True)

def adjusted_effect(shop):
    sub = claims[claims["Shop"].isin([shop] + REFERENCE)]
    total = 0
    for a in AREAS:
        g = sub[sub["Area"] == a]
        total += area_share[a] * (g[g["Shop"] == shop]["FraudRisk"].mean() - g[g["Shop"].isin(REFERENCE)]["FraudRisk"].mean())
    return total

within = (claims[claims["Shop"].isin(["Bayview Collision"] + REFERENCE)]
          .assign(Group=lambda d: np.where(d["Shop"] == "Bayview Collision", "Bayview Collision", "reference shops"))
          .groupby(["Area", "Group"])["FraudRisk"].agg(claims="size", fraud_share="mean").reset_index())
print("Within each area, Bayview Collision against the reference shops:")
print(within.to_string(index=False))
print()
print("ADJUSTING FOR AREA: extra fraud share at the shop")
print(f"  Bayview Collision: {100 * adjusted_effect('Bayview Collision'):+.1f} percentage points")
print(f"  Apex Auto Body:    {100 * adjusted_effect('Apex Auto Body'):+.1f} percentage points")

**Look at the first table.** Within Riverside, Bayview Collision is about the same as the reference shops, and within Northgate it is about the same too. The shop looked risky because **most of its customers live in Riverside**, where prior claims and fraud are more common. Once the area is held fixed, the Bayview effect is about zero. **Apex Auto Body** stays high in both areas: its effect barely moves.

### Step 3: Do it in the machine
We can also test the hypothesis directly. In a simulation, we can **force** every claim to use a given shop (`do(Shop = ...)`) and see what happens to the fraud rate, with everything else about each claim unchanged.

In [ ]:
M = 200000
U2 = draw_noise2(M, np.random.default_rng(31))
def fraud_rate_if_every_claim_uses(shop):
    return world2(U2, do={"Shop": np.full(M, shop)})["FraudRisk"].mean()

print(f"Fraud rate as things are:                {world2(U2)['FraudRisk'].mean():.4f}")
print(f"do(every claim uses Crescent Repair):    {fraud_rate_if_every_claim_uses('Crescent Repair'):.4f}")
print(f"do(every claim uses Bayview Collision):  {fraud_rate_if_every_claim_uses('Bayview Collision'):.4f}")
print(f"do(every claim uses Apex Auto Body):     {fraud_rate_if_every_claim_uses('Apex Auto Body'):.4f}")

Sending every claim to Bayview Collision gives **exactly the same fraud rate** as sending every claim to Crescent Repair. The shop has no effect. Sending every claim to Apex Auto Body raises it sharply.

### The results side by side

In [ ]:
results = pd.DataFrame({
    "Seeing (data alone)":      [100 * naive_effect("Bayview Collision"), 100 * naive_effect("Apex Auto Body")],
    "Adjusting for Area":       [100 * adjusted_effect("Bayview Collision"), 100 * adjusted_effect("Apex Auto Body")],
    "do() in the machine":      [100 * (fraud_rate_if_every_claim_uses("Bayview Collision") - fraud_rate_if_every_claim_uses("Crescent Repair")),
                                 100 * (fraud_rate_if_every_claim_uses("Apex Auto Body") - fraud_rate_if_every_claim_uses("Crescent Repair"))],
}, index=["Bayview Collision", "Apex Auto Body"])
results.columns = [c + " (points)" for c in results.columns]
results

**The machine, opened.** This is the truth we planted when we built the claims, which is the only reason we can check the answer:
- **Apex Auto Body** runs a fraud ring. Claims there are fraudulent about 85% of the time, whatever the claimant's history. That is a real effect of the shop.
- **Bayview Collision** has **no effect on fraud**. It serves Riverside, where prior claims are common (40% of claimants, against 8% in Northgate), and prior claims raise fraud.
- The six adjusters have no effect either.

Adjusting for the common cause recovered the truth for both shops: no effect for Bayview, a large one for Apex. The data alone got Bayview wrong. In real life we never get to open the machine, which is why the **graph has to be argued for**, and a confounder we did not think of would fool the adjustment too.

### Turning it into action
The graph found two shops. The causal check says what to do about each:

In [ ]:
isf = claims["FraudRisk"].values == 1
fl = claims["Flagged"].values
apex = (claims["Shop"] == "Apex Auto Body").values
bay = (claims["Shop"] == "Bayview Collision").values
missed_at_apex = int((isf & apex & ~fl).sum())

triage = pd.DataFrame([
    ("Review every claim",                          len(claims),   int(isf.sum())),
    ("Review every claim at Bayview Collision",     int(bay.sum()), int((isf & bay).sum())),
    ("Review every claim at Apex Auto Body",        int(apex.sum()), int((isf & apex).sum())),
    ("Review the claims the model flagged",         int(fl.sum()),  int((isf & fl).sum())),
    ("Review Apex claims the model did NOT flag",   int((apex & ~fl).sum()), missed_at_apex),
], columns=["Strategy", "Claims reviewed", "Fraud found"])
triage["Hit rate"] = (100 * triage["Fraud found"] / triage["Claims reviewed"]).map("{:.0f}%".format)
print(triage.to_string(index=False))
print(f"\nThe model catches {100 * fl[isf & apex].mean():.0f}% of the fraud at Apex Auto Body, so {missed_at_apex} fraudulent Apex claims got through.")

| Finding | Rung | Action |
|---|---|---|
| Apex Auto Body raises fraud, and the model misses a good share of it | 2 | **Audit the shop.** Reviewing its unflagged claims finds fraud the model let through. |
| Bayview Collision looks risky but has no effect once the area is accounted for | 2 | **Do not penalize the shop or its customers.** Reviewing all of its claims gives a hit rate only slightly above average, and closing the shop would not reduce fraud. |
| Adjusters differ only by chance | 1 | **No action.** |

The hit rate for "review every claim at Bayview" is higher than average, so it would look like a good rule. It would send honest Riverside customers, like Maria, to a slow track for a reason that is not a cause.

## Part 7: Does a false alarm cost us customers? (rung 2)

So far the question was about fraud. A business also asks about **customers**: when the model flags an honest claimant, do they renew their policy less often? And is that the *flag's* doing, or would those customers have left anyway?

We state the hypothesis and the graph first.
- **H3:** Being flagged **lowers the chance that an honest customer renews** (the slow review, the document requests, the feeling of being accused).
- **PriorClaims** has two jobs. It makes the model flag a claim (the model reads it), and it also affects renewal **for reasons of its own**: customers with a claim history tend to leave more often, flagged or not.
- **Tenure** also affects renewal, but it has nothing to do with the flag.

So there is a **backdoor path** from the flag to renewal: *Flagged ← PriorClaims → Renewal*. Honest customers who are flagged are mostly customers with a prior claim, who would renew less anyway. Reading the gap straight from the data mixes the flag's effect with the prior-claims effect. This is the same trap as the shop, with PriorClaims as the common cause.

**Who are we asking about?** Honest customers who were flagged: the people a false alarm actually touched, including Maria.

### More customers, for a steadier answer
Only 77 honest customers were flagged in this quarter's 6,000 claims, too few for a steady estimate. The insurer has the same kind of records for earlier quarters, so we use a **CRM extract of 24,000 closed claims**: this quarter's 6,000 plus 18,000 earlier ones.

In [ ]:
# ----- Earlier quarters, produced by the same machines (new random numbers) -----
N_OLD = 18000
older = world2(draw_noise2(N_OLD, np.random.default_rng(15)))
older["Flagged"] = model_flag(older)
(older["Tenure"], older["Premium"], older["WaitDays"], _, older["Renewed"]) = crm_outcomes(
    older["PriorClaims"].values, older["Flagged"].values, draw_crm_noise(N_OLD, np.random.default_rng(22)))

this_q = claims[["Area", "Shop", "PriorClaims", "FraudRisk"]].copy()
this_q["Flagged"], this_q["Tenure"], this_q["Premium"], this_q["WaitDays"], this_q["Renewed"] = flag_q, tenure, premium, wait, renewed

keep = ["Area", "Shop", "PriorClaims", "FraudRisk", "Flagged", "Tenure", "Premium", "WaitDays", "Renewed"]
crm_all = pd.concat([this_q[keep], older[keep]], ignore_index=True)
honest = crm_all[crm_all["FraudRisk"] == 0]

print(f"CRM extract: {len(crm_all):,} claims, of which {len(honest):,} legitimate.")
print(f"Legitimate customers who were flagged (false alarms): {int((honest['Flagged'] == 1).sum())}")
print(f"Overall renewal rate: {100 * crm_all['Renewed'].mean():.1f}%")

### Step 1: Look at the data (rung 1)
Compare renewal for honest customers who were flagged with honest customers who were not.

In [ ]:
flagged_h, unflagged_h = honest[honest["Flagged"] == 1], honest[honest["Flagged"] == 0]
naive_loss = 100 * (unflagged_h["Renewed"].mean() - flagged_h["Renewed"].mean())

print(f"Honest customers NOT flagged: {100 * unflagged_h['Renewed'].mean():.1f}% renewed  ({len(unflagged_h):,} customers)")
print(f"Honest customers flagged:     {100 * flagged_h['Renewed'].mean():.1f}% renewed  ({len(flagged_h):,} customers)")
print(f"\nSEEING: flagged customers renew {naive_loss:.1f} percentage points less often.")
print(f"Share with a prior claim:  flagged {100 * flagged_h['PriorClaims'].mean():.0f}%   not flagged {100 * unflagged_h['PriorClaims'].mean():.0f}%")

Flagged honest customers renew much less often. But look at the last line: **more than half of the flagged honest customers have a prior claim, against a small fraction of the unflagged**. The two groups are not alike to begin with, which is the backdoor path at work.

### Step 2: Hold the common cause fixed
Compare within each prior-claims group, then combine the groups using the mix among the **flagged** customers (because they are the people we are asking about).

In [ ]:
within_prior = (honest.groupby(["PriorClaims", "Flagged"]).agg(customers=("Renewed", "size"), renewed_pct=("Renewed", lambda s: 100 * s.mean()))
                .reset_index())
print(within_prior.to_string(index=False))

def retention_effect(df):
    f, p, r = df["Flagged"].values, df["PriorClaims"].values, df["Renewed"].values
    n_flagged = (f == 1).sum()
    total = 0.0
    for k in (0, 1):
        weight = ((f == 1) & (p == k)).sum() / n_flagged
        total += weight * (r[(f == 0) & (p == k)].mean() - r[(f == 1) & (p == k)].mean())
    return 100 * total

adj_loss = retention_effect(honest)
rng_b = np.random.default_rng(0)
boot = [retention_effect(honest.iloc[rng_b.integers(len(honest), size=len(honest))]) for _ in range(300)]
print(f"\nADJUSTING FOR PRIOR CLAIMS: flagged customers renew {adj_loss:.1f} points less often (give or take {np.std(boot):.1f})")

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.8))
x = np.arange(2); wd = 0.36
for j, (flag_val, color, lab) in enumerate([(0, "#4a90c2", "not flagged"), (1, "#d9534f", "flagged")]):
    vals = [within_prior[(within_prior["PriorClaims"] == k) & (within_prior["Flagged"] == flag_val)]["renewed_pct"].iloc[0] for k in (0, 1)]
    bars = ax.bar(x + (j - 0.5) * wd, vals, wd, color=color, label=lab)
    for b_, v_ in zip(bars, vals): ax.text(b_.get_x() + b_.get_width() / 2, v_ + 1, f"{v_:.0f}%", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(["no prior claim", "has a prior claim"])
ax.set_ylabel("% of honest customers who renewed"); ax.set_ylim(0, 100); ax.legend(title="Honest customers")
ax.set_title("Renewal by prior claims and flag status")
plt.tight_layout(); plt.show()

**Reading the picture.** In **both** groups, flagged customers renew less. So the flag does cost renewals. But customers with a prior claim renew less than customers without one even when **not** flagged, and most flagged honest customers have a prior claim. The data alone blamed the flag for **both** effects. Holding prior claims fixed separates them, and the flag's own share is smaller.

### Step 3: Do it in the machine
In the simulation we can ask the question directly. Take every honest customer the model flagged, and compute the chance they renew **with** the flag and **without** it, keeping everything else about them unchanged.

In [ ]:
M_BIG = 300000
U_big = draw_noise2(M_BIG, np.random.default_rng(41))
V_big = draw_crm_noise(M_BIG, np.random.default_rng(42))
big = world2(U_big)
hit = (big["FraudRisk"].values == 0) & (model_flag(big) == 1)             # honest customers the model flags

ten_big = np.clip(V_big["tenure"], 0.5, 25)
p_with    = p_renew(big["PriorClaims"].values, 1, ten_big)
p_without = p_renew(big["PriorClaims"].values, 0, ten_big)
true_loss = 100 * (p_without[hit] - p_with[hit]).mean()
print(f"DOING: for honest customers the model flags, removing the flag raises their chance of renewing by {true_loss:.1f} points  ({int(hit.sum()):,} simulated customers)")

pd.DataFrame({"Seeing (data alone)": [naive_loss], "Adjusting for prior claims": [adj_loss], "do() in the machine": [true_loss]},
             index=["Renewal lost to a false alarm (points)"])

**The machine, opened.** This is the truth we planted:
- Being flagged lowers the **odds** of renewing, and prior claims do too, by a similar amount, **independently** of each other.
- Tenure helps a little (long-standing customers stay).
- A fraud review adds about two weeks to the time it takes to settle.

The data alone overstated the harm of the flag, because it also carried the effect of prior claims. Adjusting for the common cause came close to the truth (the small difference is chance in a finite sample). We give a **range**, not a single number, since we only have finitely many customers.

| Finding | Rung | What it means for the business |
|---|---|---|
| Flagged customers renew less, and wait longer | 1 | Look after them; they are a segment worth watching |
| Part of the gap is the flag, part is prior claims | 2 | A false alarm has a **real, measurable cost**, smaller than the raw gap suggests |
| Maria's own renewal | 3 | Part 9 |

## Part 8: What is a policy worth? A business case

Part 6 said the model misses a good share of the fraud at Apex Auto Body. Part 7 put a price on a false alarm. Now put them together and compare **three review policies**. In each, the insurer also sends the claims that the model did **not** flag from a given group to the slow fraud review:

1. Also review every unflagged **Bayview Collision** claim.
2. Also review every unflagged **Riverside** claim.
3. Also review every unflagged **Apex Auto Body** claim.

**These dollar figures are illustrative assumptions, not facts about any real insurer.** Change them and rerun to see how the answer moves.

In [ ]:
FRAUD_SAVING   = 8000      # ASSUMPTION: dollars saved each time a fraudulent claim is caught
REVIEW_COST    = 150       # ASSUMPTION: dollars to review one extra claim
LIFETIME_YEARS = 3         # ASSUMPTION: years of premium we expect from a customer who renews

def usd(x): return f"-${abs(x):,.0f}" if x < 0 else f"${x:,.0f}"

prem = pd.Series(premium, index=claims.index)
unflagged = ~claims["Flagged"].values
is_fraud_q = claims["FraudRisk"].values == 1

def policy_row(name, group_mask):
    m = group_mask & unflagged
    n, fraud_found = int(m.sum()), int((m & is_fraud_q).sum())
    honest_hit = m & ~is_fraud_q
    value_lost = (adj_loss / 100) * prem[honest_hit].sum() * LIFETIME_YEARS      # renewals lost x premium x years
    cost = n * REVIEW_COST
    saved = fraud_found * FRAUD_SAVING
    return dict(Policy=name, ClaimsReviewed=n, FraudFound=fraud_found, HonestCustomersReviewed=int(honest_hit.sum()),
                FraudSaved=saved, ReviewCost=cost, PremiumValueLost=value_lost, Net=saved - cost - value_lost,
                BreakEvenSavingPerFraud=(cost + value_lost) / max(fraud_found, 1))

policy_table = pd.DataFrame([
    policy_row("Also review unflagged Bayview claims", (claims["Shop"] == "Bayview Collision").values),
    policy_row("Also review unflagged Riverside claims", (claims["Area"] == "Riverside").values),
    policy_row("Also review unflagged Apex claims", (claims["Shop"] == "Apex Auto Body").values),
])
money = ["FraudSaved", "ReviewCost", "PremiumValueLost", "Net", "BreakEvenSavingPerFraud"]
shown = policy_table.copy()
for c_ in money: shown[c_] = shown[c_].map(usd)
print(shown.to_string(index=False))

**How to read the table.** *FraudSaved* is the benefit. *ReviewCost* and *PremiumValueLost* are the costs: every honest customer pulled into review has a lower chance of renewing, using the effect we measured in Part 7. *Net* is the benefit minus the costs. We assume every honest customer pulled into review loses the same share of renewal that we measured in Part 7, which is a simplification. *BreakEvenSavingPerFraud* is the amount each caught fraud would have to be worth before the policy paid for itself, which does not depend on the assumed $8,000.

- **Apex** pays for itself by a wide margin. Nearly every unflagged claim reviewed is a real fraud, so few honest customers are touched.
- **Bayview** and **Riverside** send **many** honest customers through a slow review for each fraud found. Under these assumptions, they cost more than they save. The break-even column shows how large the saving per caught fraud would have to be to change that.

Part 6 said the same thing in words: penalizing Bayview does not reduce fraud, because the shop is not the cause. This table adds the cost to the customer relationship.

## Part 9: Maria's own plan (rung 3)

The shop check cleared Bayview Collision for the *group*. What about Maria herself? And what would help her?

**Her shop.** Bayview Collision does not appear in any of the equations that produced Maria's fraud status, her receipts or her witness statements. Replaying her case at a different shop would change nothing in her story: her score stays exactly as it is. The shop was part of how she got flagged in a graph picture, and no part of why her clues looked suspicious.

**Her clues.** This part is the same machine as before. We **recall** Maria's own noise (her receipts came out 1.85 worse than the equation predicts, her witness statements 2.0 worse), **change** what she can act on, and **replay**.

In [ ]:
u_receipts = maria["Receipts"] - eq_receipts(maria["FraudRisk"], maria["Docs"], 0.0)
u_witness  = maria["Witness"]  - eq_witness(maria["FraudRisk"], maria["FollowUp"], 0.0)
print(f"Maria's receipt noise: {u_receipts:+.2f}    Maria's witness noise: {u_witness:+.2f}")

def counterfactual_maria(extra_docs=0, follow_up=0):
    r = eq_receipts(maria["FraudRisk"], maria["Docs"] + extra_docs, u_receipts)   # her own noise, new documents
    w = eq_witness(maria["FraudRisk"], follow_up, u_witness)                      # her own noise, new follow-up
    x = pd.DataFrame([dict(PriorClaims=1, SuspiciousTiming=0, Receipts=r, Witness=w)])
    return model.predict_proba(x[features])[0, 1]

COST_DOC, COST_FOLLOWUP = 1, 4
plans = pd.DataFrame([dict(ExtraDocs=e, FollowUp=f, Cost=e * COST_DOC + f * COST_FOLLOWUP, Probability=counterfactual_maria(e, f))
                      for e in range(0, 9) for f in (0, 1)])
plans["Decision"] = np.where(plans["Probability"] < 0.5, "cleared", "still flagged")
print()
print(plans[plans["FollowUp"] == 0].head(8).to_string(index=False))
best = plans[plans["Probability"] < 0.5].sort_values(["Cost", "ExtraDocs"]).iloc[0]
print(f"\nCheapest plan that clears her: {int(best.ExtraDocs)} extra documents, follow-up = {int(best.FollowUp)}, cost {int(best.Cost)}")
print(f"Follow-up alone (no extra documents): probability {counterfactual_maria(0, 1):.3f}")

For a **group** of claimants like Maria, about four more documents is enough for nearly everyone. For **Maria**, who is unlucky in her own paperwork, it takes five, at a cost of 5. A follow-up on its own is not enough. That is the difference between a group answer and an individual one, and it is the part of the case file only the third rung can supply.

### Would Maria have renewed if she had not been flagged?
Maria was a customer for **6.0 years**, paid a **$1,350** premium, and her policy **lapsed**. The insurer's question is a counterfactual about one person: *if she had not been flagged, would she have stayed?*

The steps are the same three, with one difference. Renewal is a **yes or no** outcome, like the fraud status earlier, so her own hidden number (call it her **renewal roll**, a number between 0 and 1) is never recorded.

1. **Recall.** The rule is: a customer renews if her roll is **below** her chance of renewing. With the flag, Maria's chance was a little over one half. She did not renew, so her roll was **above** that chance. That is all we can say about it: it lies somewhere between her chance and 1, like a number in a sealed envelope.
2. **Change.** Remove the flag. Her chance of renewing goes up. Nothing else about her changes: same tenure, same prior claim.
3. **Replay.** With the same roll and a higher bar, would she have cleared it? We cannot say for sure. Her roll could be between the old bar and the new bar (she would have stayed), or above both (she would have left anyway).

The chance of the first case is the share of the sealed-envelope range that lies below the new bar.

In [ ]:
p_flagged  = p_renew(prior=1, flagged=1, tenure=maria_crm["Tenure"])        # her chance of renewing, as she was treated
p_unflagged = p_renew(prior=1, flagged=0, tenure=maria_crm["Tenure"])       # her chance if she had not been flagged

# Her roll is somewhere between p_flagged and 1 (she left). What share of that range is below p_unflagged?
chance_she_would_have_stayed = (p_unflagged - p_flagged) / (1 - p_flagged)

print(f"Maria's chance of renewing as she was treated (flagged):  {p_flagged:.3f}")
print(f"Her chance if she had not been flagged:                   {p_unflagged:.3f}")
print(f"Her roll lies between {p_flagged:.3f} and 1.000, and she would have stayed if it lies below {p_unflagged:.3f}.")
print(f"\nChance that Maria would have renewed without the flag: {chance_she_would_have_stayed:.0%}")

# Check by brute force: many customers like Maria, keep only those who left, and see how many would have stayed
rolls = np.random.default_rng(5).uniform(size=2_000_000)
left = rolls >= p_flagged
print(f"Check by simulation (customers like Maria who left): {100 * (rolls[left] < p_unflagged).mean():.1f}% would have stayed")

**Reading the result.** There is **no certain answer** for Maria. It is **likely but not guaranteed** that she would have stayed. Compare this with her receipts earlier: there the recalled number was fixed exactly, and the replay gave one exact answer. Here the outcome was yes or no, so recall leaves a range, and the replay gives a **probability**. A counterfactual for one person is only as sharp as the equation allows.

For the insurer, the useful reading is about the **action**: Maria is a long-standing, honest customer who was more likely than not to have been kept without the false alarm, so she is worth a personal call, not a form letter.

## Part 10: Closing the loop: test the next best action (rung 2)

The case file will recommend a **next best action** for customers like Maria: make the document requirements clear **at the start**, so honest claimants (especially those with a prior claim) do not look suspicious only because their paperwork is thin. We have a **plan**. Before changing how every claim is handled, run a **pilot** and measure it.

**The design.**
- Take **20,000 new incoming claims**.
- For each claim, a **coin flip** decides whether the claimant gets a **document checklist at intake** (the checklist adds three supporting documents) or **business as usual**.
- Measure the results for claimants **with a prior claim**, since they are the people the checklist is aimed at. In the simulation we know which claims are honest. A real insurer learns that only after a full review, so it would compare honest claimants after the fact.

Because the assignment is random, **nothing about the customers decides who gets the checklist**. There is no backdoor path, so a simple comparison of the two groups answers the question. That is why a randomized pilot is the most direct rung 2 evidence available: it **is** the do() experiment, run in the real world.

**A note on the simulation.** To check ourselves, the code computes **both** worlds for every claim. A real insurer sees only one world per claim, the one the coin flip picked. We use the other only to see how close the pilot gets to the truth.

In [ ]:
N_PILOT, CHECKLIST_DOCS = 20000, 3
U_p = draw_noise2(N_PILOT, np.random.default_rng(16))
V_p = draw_crm_noise(N_PILOT, np.random.default_rng(116))
gets_list = np.random.default_rng(216).uniform(size=N_PILOT) < 0.5             # the coin flip

as_is     = world2(U_p)
with_list = world2(U_p, do={"Docs": as_is["Docs"].values + CHECKLIST_DOCS})     # the same claims, with three more documents

def outcomes(df):
    f = model_flag(df)
    _, _, w, _, r = crm_outcomes(df["PriorClaims"].values, f, V_p)
    return dict(flagged=f.astype(float), wait=w, renewed=r.astype(float))

res = {"business as usual": outcomes(as_is), "checklist": outcomes(with_list)}
seen = {k: np.where(gets_list, res["checklist"][k], res["business as usual"][k]) for k in res["checklist"]}   # what the insurer observes
is_f = as_is["FraudRisk"].values == 1                                              # known here only because we built the claims
target = as_is["PriorClaims"].values == 1                                         # claimants with a prior claim
honest_target = target & ~is_f
fraud_target = target & is_f

def compare(label, key, subset):
    a, b = seen[key][subset & gets_list], seen[key][subset & ~gets_list]
    d = a.mean() - b.mean()
    se = np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
    truth = (res["checklist"][key][subset] - res["business as usual"][key][subset]).mean()
    return dict(Measure=label, AsUsual=b.mean(), Checklist=a.mean(), Difference=d, Low=d - 1.96 * se, High=d + 1.96 * se, Truth=truth)

pilot = pd.DataFrame([
    compare("Honest claimants wrongly flagged", "flagged", honest_target),
    compare("Real fraud that the model flags", "flagged", fraud_target),
    compare("Days to settle (honest claimants)", "wait", honest_target),
    compare("Renewed (honest claimants)", "renewed", honest_target),
])
print(f"Honest claimants with a prior claim: {int((honest_target & gets_list).sum()):,} got the checklist, {int((honest_target & ~gets_list).sum()):,} did not.")
print(f"Fraudulent claims with a prior claim: {int((fraud_target & gets_list).sum()):,} got the checklist, {int((fraud_target & ~gets_list).sum()):,} did not.\n")
print(pilot.round(3).to_string(index=False))

**Reading the pilot.** The columns are: the average with business as usual, the average with the checklist, the difference, a **range** for that difference (Low to High: the pilot's way of saying "give or take chance"), and the **truth** that we can only know because we computed both worlds.

- **Fewer false alarms and faster settlement.** The range excludes zero for both, so these effects are real. The checklist does what it was designed to do.
- **A catch.** The share of **real fraud** that the model flags also goes **down**. The model reads receipts, and extra documents improve receipts for everyone, fraudsters included. This is the gaming problem from the limits: advice that helps honest claimants also helps dishonest ones.
- **Renewal.** The range includes zero, and the estimate even has the wrong sign: the truth, which we can see only because we computed both worlds, is a small gain. So **this pilot cannot detect the effect**. That is not evidence of no effect. We expect a small effect, because fewer false alarms times the cost of a false alarm is a small number, and a small effect needs a large pilot.

Two quick calculations show how small. They use Part 7's result.

In [ ]:
fa = pilot.iloc[0]
expected_gain = -fa["Difference"] * (adj_loss / 100)                  # fewer false alarms x renewal lost per false alarm
print(f"Fewer false alarms: {-100 * fa['Difference']:.1f} points of honest claimants with a prior claim")
print(f"Each false alarm costs about {adj_loss:.1f} points of renewal, so we expect roughly {100 * expected_gain:.1f} points more renewals.")

p0 = pilot.iloc[3]["AsUsual"]
n_per_group = 2 * (1.96 + 0.84) ** 2 * p0 * (1 - p0) / expected_gain ** 2     # standard sample size for 80% power
print(f"To see an effect that small reliably, a pilot needs roughly {n_per_group:,.0f} honest claimants with a prior claim in each group.")
print(f"This pilot had {int((honest_target & gets_list).sum()):,} and {int((honest_target & ~gets_list).sum()):,}.")

**What the insurer learns, and the next step.** The pilot shows that the checklist **reduces false alarms and shortens the wait**, at the cost of **letting some real fraud look better to the model**. That points to a decision, not a verdict: roll the checklist out **together with the Apex audit** from Part 8, so the fraud the model can no longer see is still caught by a check that does not depend on receipts. Then log the new results as interactions, and run the loop again. This is the **closed loop** of CRM: **record, analyze, act, measure, record again**.

## Part 11: The case file

Now we assemble everything into one file a reviewer can act on. Each line carries the **rung** of its evidence, so nobody reads a pattern as a cause.

This is retrieval and grounding again, as in the earlier graph RAG demo: **gather the facts first, then write the answer from exactly those facts.** We gather the facts into a list, each tagged with its rung, and render the case file from it.

In [ ]:
sim_top = claims.iloc[np.argsort(-sims)[:25]]
cluster_row = summary.loc[maria_cluster]
bay_row = shop_table.set_index("shop").loc["Bayview Collision"]
apex_row = shop_table.set_index("shop").loc["Apex Auto Body"]
riv = group_table(G, "Area").set_index("area").loc["Riverside"]

facts = [
    ("DECISION", f"The model scored Maria's claim {p_maria:.3f} and flagged it. A full review found the claim legitimate (a false positive)."),
    ("RUNG 1", f"Claims whose notes read most like hers (top 25) were {100 * sim_top['FraudRisk'].mean():.0f}% fraud, against {100 * claims['FraudRisk'].mean():.0f}% overall. Her note has the wording of a fraudulent claim."),
    ("RUNG 1", f"She falls in a cluster of {int(cluster_row['claims'])} claims with prior claims, of which {100 * cluster_row['fraud_share']:.0f}% were fraud."),
    ("RUNG 1", f"Her repair shop, Bayview Collision, handled {int(bay_row['claims']):,} closed claims, {bay_row['fraud_pct']:.0f}% fraud. Her area, Riverside, is {riv['fraud_pct']:.0f}% fraud."),
    ("RUNG 2", f"Bayview Collision's effect on fraud, once the area is held fixed, is {100 * adjusted_effect('Bayview Collision'):+.1f} points. Forcing claims to Bayview changes nothing."),
    ("RUNG 2", f"Apex Auto Body raises fraud by {100 * adjusted_effect('Apex Auto Body'):+.1f} points after adjusting for area. The model misses {missed_at_apex} fraudulent claims there."),
    ("RUNG 3", f"For Maria herself, four extra documents is not enough (probability {counterfactual_maria(4):.3f}). Five clears her (probability {counterfactual_maria(5):.3f}), at a cost of {int(best.Cost)}. A follow-up alone does not (probability {counterfactual_maria(0, 1):.3f})."),
    ("RUNG 1", f"Customer record: Maria had been with the insurer {maria_crm['Tenure']:.1f} years, paid ${maria_crm['Premium']:,.0f} a year, waited {maria_crm['WaitDays']} days for her claim to settle, and did not renew. Her segment renewed {segments.loc[maria_cluster, 'renewed_pct']:.0f}% of the time."),
    ("RUNG 2", f"For honest customers the model flags, a flag lowers renewal by about {adj_loss:.1f} points (the raw gap of {naive_loss:.1f} also carries the effect of prior claims)."),
    ("RUNG 3", f"If Maria had not been flagged, she had about a {100 * chance_she_would_have_stayed:.0f}% chance of having renewed. The answer is a probability, not a certainty."),
    ("RUNG 2", f"Under stated assumptions, reviewing unflagged Apex claims nets {usd(policy_table.iloc[2]['Net'])}, while the same step for Bayview nets {usd(policy_table.iloc[0]['Net'])} and for Riverside {usd(policy_table.iloc[1]['Net'])}."),
    ("RUNG 2", f"A pilot of a document checklist at intake cut false alarms among honest claimants with a prior claim by {-100 * pilot.iloc[0]['Difference']:.1f} points and their wait by {-pilot.iloc[2]['Difference']:.1f} days, and it also lowered the share of real fraud the model flags by {-100 * pilot.iloc[1]['Difference']:.1f} points. Its effect on renewal was too small for this pilot to detect."),
]

def render_case_file(facts):
    out = ["CASE FILE: CLAIM 'Maria'", "=" * 74]
    for tag, text in facts:
        out.append(f"[{tag:8s}] {text}")
    out += ["-" * 74, "RECOMMENDED ACTIONS",
            "  For Maria:   Ask for five additional supporting documents, then move the claim to the fast track.",
            "               Do not let her shop or area count against her.",
            "               Have a person call her about her renewal. Log the call as an interaction.",
            "  For the insurer: Audit Apex Auto Body, starting with its unflagged claims.",
            "               Do not penalize Bayview Collision or Riverside claimants as a group.",
            "               Roll out the document checklist together with the Apex audit, then measure again.",
            "  Watch:       If this advice becomes public, everyone can add documents. See Part 12."]
    return "\n".join(out)

print(render_case_file(facts))

### Optional: let a language model write the summary
As in the earlier graph RAG demo, the model is told to use **only** the facts we retrieved. We add one more rule: it may never describe a `RUNG 1` fact as a cause. If no Groq key is found, the cell prints a fallback message and everything above still stands.

In [ ]:
MODEL = "openai/gpt-oss-120b"

def build_grounded_prompt(facts):
    facts_text = "\n".join(f"- [{tag}] {text}" for tag, text in facts)
    system_message = (
        "You are a careful assistant writing a short case summary for an insurance reviewer. "
        "Use ONLY the facts listed below. Each fact is tagged with the rung of evidence behind it: "
        "RUNG 1 is association, RUNG 2 is the tested effect of an intervention on a group, "
        "RUNG 3 is a counterfactual for one individual. Never describe a RUNG 1 fact as a cause. "
        "If the facts do not support a statement, say so.\n\nFACTS:\n" + facts_text)
    return [{"role": "system", "content": system_message},
            {"role": "user", "content": "Write a five-sentence summary of this case and recommend what the reviewer should do."}]

def ask_model(messages, model=MODEL):
    if not GROQ_API_KEY:
        return "[OFFLINE FALLBACK -- no GROQ_API_KEY found. Add your key to .env to see the model's summary.]"
    try:
        import groq
        client = groq.Groq(api_key=GROQ_API_KEY)
        resp = client.chat.completions.create(model=model, messages=messages, temperature=0.2)
        return resp.choices[0].message.content
    except Exception as e:
        return f"[Groq API error: {e}]"

print(ask_model(build_grounded_prompt(facts)))

## Part 12: Honest limits

Everything above worked because we planted the truth and could check it. Seven cautions for real data:

1. **Similarity, clusters and graph structure are rung 1.** They describe what goes with what. A shop with a high fraud share, a cluster with a high fraud share and a note that looks like fraud are all warnings to investigate, and not findings about causes.
2. **The graph for rung 2 is an assumption.** We got the right answer because Area was the common cause and we held it fixed. If another common cause we never recorded had been behind Bayview's numbers, adjusting for Area would not have caught it. Recourse and shop effects are advice conditional on a causal story.
3. **Guilt by association.** Treating every Bayview claim or every Riverside claimant as suspicious would have sent honest people to a slow track for a reason that is not a cause. That is a fairness problem as well as an accuracy problem.
4. **Advice that everyone follows can weaken the signal.** If the insurer tells every flagged claimant to submit five more documents, honest and dishonest claimants alike will do it, and receipts stop telling the model much. Our own pilot showed the first sign of this: the checklist lowered the share of real fraud the model flags. The more useful and public the advice, the less informative the clue.
5. **A language model can overstate.** A written summary is fluent whether or not it is careful. Tagging each fact with its rung, and telling the model never to turn a rung 1 fact into a cause, is a guard and not a guarantee. A person still reads the case file.
6. **A customer graph is personal data.** Linking named people to their shops, their areas, their claims and every call we made about them is powerful, and it carries duties: collect only what you need, say what it is used for, let people see and correct their record, protect it, and delete it when the purpose ends. The more joined-up the record, the more a leak or a misuse can reveal.
7. **Retention can slide into unfair treatment.** A model that decides who is worth keeping can quietly treat valuable customers better and everyone else worse (a faster review, a friendlier call). A customer's worth to the insurer is not a reason for a slower or harsher process for someone else. Check who benefits from each action, not just the average. Also remember that the dollar figures in the business case were stated assumptions, and a different set can change which policy wins.

## Wrap-up

| Step | Tool | Rung | What it gave us | What it could not say |
|---|---|---|---|---|
| Represent and retrieve | TF-IDF similarity (and embeddings) | 1 | Claims that read like Maria's were mostly fraud | A look-alike is not the same case, and not a cause |
| Cluster | k-means | 1 | Maria sits in a group that is about half fraud | Says what claims look like, not why |
| Knowledge graph | NetworkX and Neo4j (Cypher) | 1 | Two shops stand out; her shop is one of them | A well-connected or flagged node is not a cause |
| Hypothesis check | Graph, adjustment, do() | 2 | Apex has a real effect; Bayview has none | Depends on the graph being right |
| Individual plan | Counterfactual and recourse | 3 | For Maria, five extra documents | Needs equations, so needs more assumptions |
| Customer record and segments | Tables, k-means | 1 | Flagged groups wait longer and renew less | Cannot say whether the flag is the reason |
| Retention | Adjustment, do() | 2 | A false alarm costs renewals, less than the raw gap suggests | Depends on the graph being right |
| Business case | Cost table | 2 | Apex audit pays; reviewing Bayview or Riverside does not (under stated assumptions) | The dollar figures are assumptions |
| Maria's renewal | Counterfactual | 3 | A likely, not certain, chance she would have stayed | A yes or no outcome leaves a range |
| Closed loop | Randomized pilot | 2 | The checklist cuts false alarms and waits, at a cost in fraud the model sees | Too small to see the effect on renewal |
| Case file | Grounded retrieval | all | One page where each line carries its rung | Still needs a person to judge it |

**The takeaways**
1. **Retrieval and clustering find look-alikes, and look-alikes are not causes.** They live on the first rung.
2. A **knowledge graph** records who is connected to whom, and Cypher lets you ask questions by drawing the pattern you want.
3. A pattern in the graph is a **hypothesis**. To test it, use the causal graph, hold the common cause fixed, and check with do().
4. A shop (or any node) can look guilty because it sits behind a **confounder**. Adjusting for the common cause recovered the truth here.
5. Group answers and individual answers differ. For Maria, the group needs four documents and she needs five, and her chance of having stayed is a probability, not a certainty.
6. **Ground the write-up in retrieved facts, and tag each with its rung**, so a reader can see what each claim rests on.
7. **CRM is the same workflow pointed at customers:** a customer record and an interaction log, segments from clustering, a relationship map in the graph, a customer 360 view, a next best action, and a pilot that closes the loop.
8. A false alarm has a **price**, and you can measure it with the same tools: hold the common cause fixed, and test with a randomized pilot when you can.
9. The workflow is only as good as its assumptions, and the people it touches deserve to know which ones were made, including how their data is used.

## Reflection (activity)
Start a new text cell and write about **100–150 words** responding to this prompt:

*Think of a system that flags people or items using similarity, groups or a network of connections (fraud, hiring, content moderation, a recommendation). Describe one pattern it might find. Then say what extra evidence you would need before treating that pattern as a cause, and one way acting on it could harm people who did nothing wrong.*